# Ajustar cinco modelos cinéticos y comparar la proyección del banco

**Pregunta:** ¿Cómo afecta un cambio de colector a la cinética ajustada y al error de proyección del banco?

Nivel: 2. Requisitos previos: Nivel 1: Interpreta un banco de flotación como respuesta cinética. Tiempo de lectura: 30 minutos.



## Objetivos

Ejecutar los ajustes no lineales acotados de OreFlow, inspeccionar límites y residuales, y comparar la predicción de banco con la referencia exacta de tasas distribuidas del motor.

## Preparación

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

El pipeline OreFlow se importa desde `data-pipeline/`; este notebook nunca instala paquetes.

Para la lección de nivel 2 sobre 02-flotation, registra las versiones fijadas antes de ejecutar el análisis; así se puede reconstruir el entorno que produjo cada tabla y figura.

In [ ]:
# Importar herramientas de análisis y registrar sus versiones exactas.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Datos

Responsable: autores de OreFlow, licencia MIT. pipeline.cases.catalog aporta el circuito de cobre y molibdeno. El motor calcula alimentación rougher, tasas y residencia; tiempos batch y ajustes numéricos provienen de engine/data/constants.json. Los registros nominales, de banco corto y dosis modificada se generan cuando la lección los usa. Datos virtuales y referencia exacta comparten supuestos del simulador, sin ser mediciones de laboratorio. Fuente: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Método

Los datos batch virtuales mezclan respuestas de primer orden por clase de partícula. OreFlow ajusta formas de primer orden, Kelsall, Klimpel, gamma y exponencial estirada con Levenberg–Marquardt acotado. Luego proyecta cada modelo a un banco; `kinetic_record` es la entrada de producción. Ajuste y proyección son verificaciones diferentes.

Llamar al ajustador de producción en un caso resuelto. Esto ejecuta las cinco implementaciones de modelos OreFlow.

In [ ]:
# Llamar al ajustador de producción en un caso resuelto.
from pipeline.cases.catalog import CASES
from pipeline.engine.circuit import simulate
from pipeline.engine.kinetics import kinetic_record

case = next((item for item in CASES if item.id == "copper_molybdenum"))
result = simulate(case.ore, case.plant, case.nominal)
fits = kinetic_record(
    result.flotation, result.ore, case.nominal.rougher_cells, result.metrics["rougher_recovery_pct"]
)
print("estado:", fits["status"], "| modelos ajustados:", len(fits.get("models", [])))


Comparar RMSE, convergencia y estado de límites; un parámetro en su límite no se interpreta como un óptimo interior.

In [ ]:
# Comparar RMSE, convergencia y estado de límites; un parámetro en su límite no se interpreta como
# un óptimo interior.
summary = pd.DataFrame(
    [
        {
            "modelo": {
                "first_order": "Primer orden",
                "kelsall": "Kelsall",
                "klimpel": "Klimpel",
                "gamma": "Gamma",
                "stretched_exponential": "Exponencial estirada",
            }.get(m["id"], m["id"]),
            "RMSE (puntos)": m["rmse_pct"],
            "converged": m["converged"],
            "en límite": ", ".join(m["at_bound"]) or "no",
        }
        for m in fits["models"]
    ]
)
display(summary.set_index("modelo").round(4))


Superpone las curvas ajustadas a las observaciones batch virtuales en los mismos tiempos.

In [ ]:
# Superpone las curvas ajustadas a las observaciones batch virtuales en los mismos tiempos.
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.scatter(
    fits["times_min"],
    fits["batch_recovery_pct"],
    color="black",
    zorder=4,
    label="Datos batch virtuales",
)
for model in fits["models"]:
    ax.plot(
        fits["times_min"],
        model["fitted_pct"],
        marker=".",
        label={
            "first_order": "Primer orden",
            "kelsall": "Kelsall",
            "klimpel": "Klimpel",
            "gamma": "Gamma",
            "stretched_exponential": "Exponencial estirada",
        }.get(model["id"], model["id"]),
    )
ax.set(
    xlabel="Tiempo batch (min)",
    ylabel="Recuperación (%)",
    title="Cinco formas ajustadas al mismo\nregistro batch",
)
ax.grid(True, alpha=0.25)
ax.legend(ncol=2, **legend_style)
fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.72)


Una superposición estrecha sobre observaciones batch virtuales demuestra ajuste en la ventana temporal elegida. No determina toda la cola de tasas, por lo que proyecciones y recuperación última pueden diferir aun con curvas similares. Inspeccionar convergencia y parámetros en límites antes de interpretar parámetros adicionales como detalle útil. Las observaciones proceden del mismo motor y carecen de ruido de medición de laboratorio.

Comparar las proyecciones agregadas del banco con el cálculo exacto de tasas distribuidas, incluso después de la última muestra batch.

In [ ]:
# Comparar las proyecciones agregadas del banco con el cálculo exacto de tasas distribuidas,
# incluso después de la última muestra batch.
projection = pd.DataFrame(
    [
        {
            "modelo": {
                "first_order": "Primer orden",
                "kelsall": "Kelsall",
                "klimpel": "Klimpel",
                "gamma": "Gamma",
                "stretched_exponential": "Exponencial estirada",
            }.get(m["id"], m["id"]),
            "proyección al banco (%)": m["bank_projection_pct"],
            "error frente al exacto (puntos)": m["lumping_error_pct"],
            "brecha asintótica (puntos)": m["ultimate_gap_pct"],
        }
        for m in fits["models"]
    ]
)
display(projection.round(3))
print("recuperación exacta del banco (%):", fits["bank"]["exact_true_flotation_pct"])


Reconstruir el ajuste nominal como ancla del experimento de reactivo. Cada modelo recibe exactamente los mismos tiempos y recuperaciones, de modo que el error refleja su forma funcional. Un modelo flexible puede describir la ventana observada y representar mal una cola lenta. Esa diferencia importa al proyectar a celdas continuas, donde la residencia pondera clases de tasa de otra manera que la exposición batch.

In [ ]:
# Recalcular el rougher de cobre y molibdeno nominal y ajustar su respuesta batch virtual con la
# rutina cinética de producción.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.engine.kinetics import kinetic_record

case = CASE_BY_ID["copper_molybdenum"]
nominal = simulate(case.ore, case.plant, case.nominal)
fits = kinetic_record(
    nominal.flotation,
    nominal.ore,
    case.nominal.rougher_cells,
    nominal.metrics["rougher_recovery_pct"],
)
display(
    pd.DataFrame(
        [
            {
                "Modelo": {
                    "first_order": "Primer orden",
                    "kelsall": "Kelsall",
                    "klimpel": "Klimpel",
                    "gamma": "Gamma",
                    "stretched_exponential": "Exponencial estirada",
                }.get(m["id"], m["id"]),
                "RMSE batch (pp)": m["rmse_pct"],
                "Error de banco (pp)": m["lumping_error_pct"],
            }
            for m in fits["models"]
        ]
    )
)


Acortar primero el banco como control geométrico. Mantener química fija permite comparar residencia disponible. La recuperación rougher del motor conserva arrastre y otras contribuciones; no se introduce directamente en una ecuación de flotación verdadera. El registro cinético entregará la referencia correspondiente en el paso siguiente. Identificar por separado la respuesta global y el componente cinético aislado.

In [ ]:
# Reducir en una la cantidad de celdas continuas sin cambiar alimentación ni dosis de reactivo.
short_point = case.nominal.with_values(rougher_cells=max(1, case.nominal.rougher_cells - 1))
short = simulate(case.ore, case.plant, short_point)
print("Recuperación del banco corto (%):", round(short.metrics["rougher_recovery_pct"], 3))


Calcular proyecciones del banco corto con la misma familia nominal. El contraste prueba si la calidad del ajuste se mantiene al cambiar residencia sin usar otro conjunto mineral. La referencia proviene de tasas conocidas por clase. Examinar si el menor residuo batch entrega también menor residuo de banco y conservar signos al diagnosticar sobrepredicción o subpredicción, en vez de guardar solo un nombre ganador.

In [ ]:
# Ajustar la misma familia cinética al banco acortado y proyectar cada ajuste sobre su geometría.
short_fits = kinetic_record(
    short.flotation, short.ore, short_point.rougher_cells, short.metrics["rougher_recovery_pct"]
)
print(
    "Recuperación exacta del banco corto (%):",
    round(short_fits["bank"]["exact_true_flotation_pct"], 3),
)


Cambiar diez por ciento la dosis y volver a la geometría nominal. Los cálculos nominal, banco corto y dosis aumentada aíslan dos mecanismos. La dosis modifica colección; la cantidad de celdas cambia exposición. Más colector puede aumentar ganga y no establece calidad del concentrado. Hace falta resolver el circuito: ajustar escalarmente una recuperación omitiría acoplamiento mineral y de agua.

In [ ]:
# Aumentar diez por ciento la dosis de colector y volver a ejecutar el circuito completo.
dose_point = case.nominal.with_values(collector_gpt=case.nominal.collector_gpt * 1.1)
dosed = simulate(case.ore, case.plant, dose_point)
print("Recuperación con dosis modificada (%):", round(dosed.metrics["rougher_recovery_pct"], 3))


Reajustar todas las formas implementadas a la respuesta con dosis modificada y proyectarlas al banco correspondiente. Reusar parámetros anteriores preguntaría por extrapolación del ajuste viejo, no por desempeño de las aproximaciones sobre la respuesta nueva. Cada forma recibe las mismas observaciones y referencia exacta. La tabla conserva dos errores en puntos porcentuales. Revisar límites y convergencia antes de atribuir significado físico a una ventaja numérica pequeña.

In [ ]:
# Proyectar las formas ajustadas con la nueva condición de colector y conservar, para cada forma,
# el error batch y el error de banco.
dose_fits = kinetic_record(
    dosed.flotation, dosed.ore, dose_point.rougher_cells, dosed.metrics["rougher_recovery_pct"]
)
errors = pd.DataFrame(
    [
        {
            "Modelo": {
                "first_order": "Primer orden",
                "kelsall": "Kelsall",
                "klimpel": "Klimpel",
                "gamma": "Gamma",
                "stretched_exponential": "Exponencial estirada",
            }.get(m["id"], m["id"]),
            "RMSE batch (pp)": m["rmse_pct"],
            "Error de banco (pp)": abs(m["lumping_error_pct"]),
        }
        for m in dose_fits["models"]
    ]
)
display(errors.round(4))


Graficar RMSE batch junto al error absoluto de banco después de cambiar dosis. La unidad vertical común permite comparar magnitudes, pero son tareas distintas: ajustar tiempos y proyectar residencia. El absoluto sirve para ordenar magnitudes; el signo sigue disponible en el registro. Ganar una tarea no significa ganar automáticamente la otra.

In [ ]:
# Comparar errores de ajuste batch y proyección al banco continuo con el cambio controlado de
# colector..
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
fig, ax = plt.subplots(figsize=(6.4, 3.8))
errors.set_index("Modelo").plot.barh(ax=ax)
ax.set(xlabel="Error de\nrecuperación\n(pp)", ylabel="", title="Colector +10%: ajuste y banco")
ax.tick_params(axis="x", rotation=0)
ax.legend(["RMSE batch", "|Error banco|"], fontsize=18, ncol=2, **legend_style)
fig.subplots_adjust(left=0.4, right=0.96, bottom=0.23, top=0.72)


La separación de cada par de barras informa sobre portabilidad de la aproximación. Ajustes batch casi superpuestos pueden proyectar bancos distintos, especialmente cuando clases lentas aportan fuera de la ventana observada. Datos y referencia comparten supuestos del motor, lo que aísla error de representación y excluye ruido de laboratorio y deriva de planta. La lección siguiente se centra en ese cambio de orden, sin presentar más barridos de dosis como validación independiente.

El registro conserva convergencia y parámetros en límites. Una tasa casi nula o una fracción rápida/lenta en su frontera puede hacer que una forma rica se comporte como otra simple. Esa degeneración importa fuera del último tiempo batch: un ajuste visualmente excelente puede admitir varias combinaciones equivalentes. Leer límites junto a brecha de recuperación última y error de banco, sin seleccionar solo por RMSE batch. En laboratorio también haría falta un modelo de error de muestreo antes de considerar importantes diferencias pequeñas de residuos. Una advertencia de convergencia es información del ajuste, no evidencia automática de mala física ni permiso para descartar el modelo sin revisar sus predicciones.

Calcular diagnósticos emparejados nominales y con dosis aumentada para cada forma. La dosis cambia diez por ciento con geometría nominal. Se conservan signo y magnitud de errores; una diferencia pequeña no demuestra una mejora estadística o física del reactivo.

In [ ]:
# Calcular diagnósticos emparejados nominales y con dosis aumentada para cada forma.
nominal_by_id = {m["id"]: m for m in fits["models"]}
dose_by_id = {m["id"]: m for m in dose_fits["models"]}
dose_comparison = pd.DataFrame(
    [
        {
            "Modelo": name,
            "RMSE batch nominal (pp)": nominal_by_id[name]["rmse_pct"],
            "RMSE batch con dosis modificada (pp)": dose_by_id[name]["rmse_pct"],
            "Error nominal de banco con signo (pp)": nominal_by_id[name]["lumping_error_pct"],
            "Error de banco con dosis modificada y signo (pp)": dose_by_id[name][
                "lumping_error_pct"
            ],
            "Cambio de RMSE batch (pp)": dose_by_id[name]["rmse_pct"]
            - nominal_by_id[name]["rmse_pct"],
        }
        for name in nominal_by_id
    ]
)
display(dose_comparison.round(4))


## Resultados

**Respuesta:** Al aumentar colector 10%, el RMSE batch de Kelsall pasa de <!-- result:RMSE batch de Kelsall (pp) -->0<!-- /result --> a <!-- result:RMSE batch de Kelsall con dosis modificada (pp) -->0<!-- /result --> pp y su error absoluto de banco de <!-- result:Error de banco de Kelsall (pp) -->0<!-- /result --> a <!-- result:Error de banco de Kelsall con dosis modificada (pp) -->0<!-- /result --> pp. La recuperación rougher global pasa de <!-- result:Recuperación global nominal (%) -->0<!-- /result --> a <!-- result:Recuperación global con dosis modificada (%) -->0<!-- /result --> por ciento. Son cambios calculados de respuesta y aproximación, no beneficios del reactivo validados independientemente.

Combinar recuperación y errores emparejados nominales y con dosis aumentada para responder cuantitativamente al cambio de reactivo.

In [ ]:
# Resumir las salidas calculadas que sostienen la interpretación de esta lección.
kinetic_by_name = {model["id"]: model for model in fits["models"]}
answer_values = {
    "Recuperación global del banco corto (%)": short.metrics["rougher_recovery_pct"],
    "Recuperación verdadera del banco corto (%)": short_fits["bank"]["exact_true_flotation_pct"],
    "RMSE batch de primer orden (pp)": kinetic_by_name["first_order"]["rmse_pct"],
    "RMSE batch de Kelsall (pp)": kinetic_by_name["kelsall"]["rmse_pct"],
    "Error de banco de primer orden (pp)": abs(kinetic_by_name["first_order"]["lumping_error_pct"]),
    "Error de banco de Kelsall (pp)": abs(kinetic_by_name["kelsall"]["lumping_error_pct"]),
}
dose_by_id = {m["id"]: m for m in dose_fits["models"]}
answer_values.update(
    {
        "RMSE batch de Kelsall con dosis modificada (pp)": dose_by_id["kelsall"]["rmse_pct"],
        "Error de banco de Kelsall con dosis modificada (pp)": abs(
            dose_by_id["kelsall"]["lumping_error_pct"]
        ),
        "RMSE batch de primer orden con dosis modificada (pp)": dose_by_id["first_order"][
            "rmse_pct"
        ],
        "Recuperación global con dosis modificada (%)": dosed.metrics["rougher_recovery_pct"],
        "Recuperación global nominal (%)": nominal.metrics["rougher_recovery_pct"],
    }
)
answer_values = {label: round(float(number), 3) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Resumen

El ajustador del producto permite distinguir residual, error de proyección y brecha de extrapolación. Toda la comparación permanece dentro de una prueba virtual generada por el motor.

## Referencias

- https://doi.org/10.1016/0301-7516(81)90019-3
- https://doi.org/10.1016/0301-7516(81)90019-3
- https://doi.org/10.1016/0301-7516(81)90019-3